<div style="border-top: 3px solid #333; border-bottom: 3px solid #333; padding: 18px 0; margin-bottom: 10px;">
<h1 style="font-size:1.6em; margin:0 0 6px 0;">Replication Study: Dropout — A Simple Way to Prevent Neural Networks from Overfitting</h1>
<p style="margin:2px 0;"><strong>Author:</strong> Belyagoubi Mohammed Abdelilah</p>
<p style="margin:2px 0;"><strong>Date:</strong> May 2026</p>
<p style="margin:2px 0;"><strong>Original Paper:</strong> Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., &amp; Salakhutdinov, R. (2014). Dropout: A Simple Way to Prevent Neural Networks from Overfitting. <em>Journal of Machine Learning Research</em>, 15(56), 1929–1958.</p>
</div>

---

## 1. Abstract

This notebook presents an empirical replication of the Dropout regularization technique proposed by Srivastava et al. (2014). Two Multilayer Perceptron (MLP) architectures of identical capacity are trained in parallel on a deliberately constrained, high-noise binary classification dataset to induce overfitting in the baseline model. The experiment isolates the effect of stochastic unit omission during training, demonstrating that Dropout forces the network to learn redundant, distributed representations that generalize significantly better to unseen data. Training and evaluation losses are tracked across all epochs to provide a direct empirical comparison.

---

## 2. Theoretical Background

### 2.1 Standard Forward Pass

In a conventional feedforward neural network, the activations at layer $l$ are computed as:

$$
y^{(l)} = f\!\left(W^{(l)}\, y^{(l-1)} + b^{(l)}\right)
$$

where $W^{(l)}$ and $b^{(l)}$ are the weight matrix and bias vector at layer $l$, and $f(\cdot)$ is a nonlinear activation function.

### 2.2 Forward Pass with Dropout

When Dropout is applied, each hidden unit is independently retained with probability $p$ (the **keep probability**) by sampling a binary mask from a Bernoulli distribution:

$$
r_j^{(l)} \sim \text{Bernoulli}(p)
$$

The activations from the previous layer are then element-wise masked:

$$
\tilde{y}^{(l-1)} = r^{(l-1)} \odot y^{(l-1)}
$$

The masked activations are propagated through the layer:

$$
y^{(l)} = f\!\left(W^{(l)}\, \tilde{y}^{(l-1)} + b^{(l)}\right)
$$

### 2.3 Inverted Dropout (PyTorch Implementation)

PyTorch implements **inverted Dropout**: during training, retained activations are scaled by $\frac{1}{p}$ to preserve the expected magnitude of the output. This eliminates the need for any weight rescaling at inference time, so evaluation proceeds on the full unmodified network.

---

## 3. Experimental Setup

| Parameter | Value |
|---|---|
| Dataset | `make_moons` (n = 1000, noise = 0.3) |
| Train / Test split | 100 / 900 samples (stratified) |
| Hidden units per layer | 200 |
| Number of hidden layers | 2 |
| Dropout keep probability $p$ | 0.5 (applied after each hidden layer) |
| Optimizer | Adam ($\alpha$ = 0.01) |
| Loss function | Binary Cross-Entropy with Logits |
| Training epochs | 500 |
| Random seed | 42 |

---

In [ ]:
# Cell 2: Imports & Reproducibility
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)

In [ ]:
# Cell 3: Dataset Generation
X, y = make_moons(n_samples=1000, noise=0.3, random_state=SEED)

# Force overfitting: only 100 train samples, 900 test samples
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.9,
    stratify=y,
    random_state=SEED
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
y_test  = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

In [ ]:
# Cell 4: Visualize Data
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=25, edgecolor="k", alpha=0.8)
plt.title("make_moons(n_samples=1000, noise=0.3)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

In [ ]:
# Cell 5: Model Definitions
class StandardMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 200),
            nn.ReLU(),
            nn.Linear(200, 200),
            nn.ReLU(),
            nn.Linear(200, 1)
        )

    def forward(self, x):
        return self.net(x)


class DropoutMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 200),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(200, 200),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(200, 1)
        )

    def forward(self, x):
        return self.net(x)


model_A = StandardMLP()
model_B = DropoutMLP()

print(model_A)
print()
print(model_B)

In [ ]:
# Cell 6: Training Setup
criterion = nn.BCEWithLogitsLoss()

optimizer_A = optim.Adam(model_A.parameters(), lr=0.01)
optimizer_B = optim.Adam(model_B.parameters(), lr=0.01)

EPOCHS = 500

train_loss_A, test_loss_A = [], []
train_loss_B, test_loss_B = [], []

train_acc_A, test_acc_A = [], []
train_acc_B, test_acc_B = [], []

In [ ]:
# Cell 7: Training Loop
def accuracy_from_logits(logits, y_true):
    preds = (torch.sigmoid(logits) >= 0.5).float()
    return (preds == y_true).float().mean().item()


for epoch in range(EPOCHS):
    # -------------------------
    # Train Standard Model A
    # -------------------------
    model_A.train()
    optimizer_A.zero_grad()

    logits_A = model_A(X_train)
    loss_A = criterion(logits_A, y_train)

    loss_A.backward()
    optimizer_A.step()

    # -------------------------
    # Train Dropout Model B
    # -------------------------
    model_B.train()   # Dropout ON
    optimizer_B.zero_grad()

    logits_B = model_B(X_train)
    loss_B = criterion(logits_B, y_train)

    loss_B.backward()
    optimizer_B.step()

    # -------------------------
    # Evaluate both models
    # -------------------------
    model_A.eval()
    model_B.eval()    # Dropout OFF

    with torch.no_grad():
        train_logits_A = model_A(X_train)
        test_logits_A  = model_A(X_test)

        train_logits_B = model_B(X_train)
        test_logits_B  = model_B(X_test)

        train_loss_A.append(criterion(train_logits_A, y_train).item())
        test_loss_A.append(criterion(test_logits_A, y_test).item())

        train_loss_B.append(criterion(train_logits_B, y_train).item())
        test_loss_B.append(criterion(test_logits_B, y_test).item())

        train_acc_A.append(accuracy_from_logits(train_logits_A, y_train))
        test_acc_A.append(accuracy_from_logits(test_logits_A, y_test))

        train_acc_B.append(accuracy_from_logits(train_logits_B, y_train))
        test_acc_B.append(accuracy_from_logits(test_logits_B, y_test))

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch+1:3d}/{EPOCHS} | "
            f"A Train/Test Loss: {train_loss_A[-1]:.4f}/{test_loss_A[-1]:.4f} | "
            f"B Train/Test Loss: {train_loss_B[-1]:.4f}/{test_loss_B[-1]:.4f}"
        )

In [ ]:
# Cell 8: Loss Comparison Plot
plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Left: standard model
axes[0].plot(train_loss_A, label="Train Loss", color="#0f766e", linewidth=2)
axes[0].plot(test_loss_A, label="Test Loss", color="#be123c", linewidth=2)
axes[0].set_title("Without Regularization (Overfitting)", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend(frameon=False)

# Right: dropout model
axes[1].plot(train_loss_B, label="Train Loss", color="#0f766e", linewidth=2)
axes[1].plot(test_loss_B, label="Test Loss", color="#2563eb", linewidth=2)
axes[1].set_title("With Dropout (Generalized)", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].legend(frameon=False)

fig.suptitle("Dropout Regularization on Noisy make_moons Data", fontsize=14, fontweight="bold")
plt.tight_layout()
os.makedirs("assets", exist_ok=True)
plt.savefig("assets/loss_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 9: Final Metrics
print(f"Standard MLP  -> Final Train Loss: {train_loss_A[-1]:.4f}")
print(f"Standard MLP  -> Final Test  Loss: {test_loss_A[-1]:.4f}")
print(f"Standard MLP  -> Final Train Acc : {train_acc_A[-1]:.3f}")
print(f"Standard MLP  -> Final Test  Acc : {test_acc_A[-1]:.3f}")
print()

print(f"Dropout MLP   -> Final Train Loss: {train_loss_B[-1]:.4f}")
print(f"Dropout MLP   -> Final Test  Loss: {test_loss_B[-1]:.4f}")
print(f"Dropout MLP   -> Final Train Acc : {train_acc_B[-1]:.3f}")
print(f"Dropout MLP   -> Final Test  Acc : {test_acc_B[-1]:.3f}")

In [ ]:
# Cell 10: Decision Boundary Plots
def plot_decision_boundary(model, X, y, title):
    model.eval()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

    with torch.no_grad():
        probs = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, probs, levels=30, cmap="coolwarm", alpha=0.5)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=20, edgecolor="k")
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    # Save the decision boundary graph
    filename = f"decision_boundary_{title.lower().replace(' ', '_').replace('-', '')}.png"
    import re
    filename = re.sub(r'_+', '_', filename)
    plt.savefig(os.path.join("assets", filename), dpi=160, bbox_inches="tight")
    
    plt.show()

plot_decision_boundary(model_A, X, y, "Decision Boundary - Standard MLP")
plot_decision_boundary(model_B, X, y, "Decision Boundary - Dropout MLP")